# Building Complex Models sing the Functional API

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# IMPORT DATASET

In [2]:
import requests

url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
output_path = "housing.csv"

response = requests.get(url)
response.raise_for_status()  # Raise an exception for HTTP errors

with open(output_path, "wb") as f:
    f.write(response.content)

print(f"Downloaded {output_path}")

Downloaded housing.csv


In [3]:
housing = pd.read_csv('housing.csv')
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
housing.drop(columns="ocean_proximity", axis=1, inplace=True)

In [5]:
housing.dropna(axis=0, inplace=True)

# TRAIN TEST SET SPLIT

In [6]:
X_train_full, X_test,y_train_full, y_test = train_test_split(housing.drop('median_house_value', axis=1), housing['median_house_value'], random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full)

## X Scale

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

Y Scale

In [8]:
scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(np.array(y_train).reshape(-1, 1))
y_valid = scaler_y.transform(np.array(y_valid).reshape(-1, 1))
y_test = scaler_y.transform(np.array(y_test).reshape(-1, 1))

# Building Complex Models sing the Functional API

In [9]:
input_ = keras.layers.Input(shape=X_train.shape[1:])
hidden1 = keras.layers.Dense(30, activation='relu')(input_)
hidden2 = keras.layers.Dense(30, activation='relu')(hidden1)
concat = keras.layers.Concatenate()([input_, hidden2])
output = keras.layers.Dense(1)(concat)
model = keras.models.Model(inputs=[input_], outputs=[output])

In [10]:
input_A = keras.layers.Input(shape=[5], name="wide_input")
input_B = keras.layers.Input(shape=[6], name="deep_input")
hidden1 = keras.layers.Dense(30, activation="relu")(input_B)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
concat = keras.layers.concatenate([input_A, hidden2])
output = keras.layers.Dense(1, name="output")(concat)
model = keras.Model(inputs=[input_A, input_B], outputs=[output])

## MODEL TRAINING

In [11]:
model.compile(loss="mse", optimizer=keras.optimizers.SGD(learning_rate=1e-3))
X_train_A, X_train_B = X_train[:, :5], X_train[:, 2:]
X_valid_A, X_valid_B = X_valid[:, :5], X_valid[:, 2:]
X_test_A, X_test_B = X_test[:, :5], X_test[:, 2:]
X_new_A, X_new_B = X_test_A[:3], X_test_B[:3]
history = model.fit((X_train_A, X_train_B), y_train, epochs=20,
validation_data=((X_valid_A, X_valid_B), y_valid))
mse_test = model.evaluate((X_test_A, X_test_B), y_test)
y_pred = model.predict((X_new_A, X_new_B))

Epoch 1/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9690 - val_loss: 0.7173
Epoch 2/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6539 - val_loss: 0.5294
Epoch 3/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.4915 - val_loss: 0.4393
Epoch 4/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.4225 - val_loss: 0.4058
Epoch 5/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3883 - val_loss: 0.3920
Epoch 6/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3703 - val_loss: 0.3829
Epoch 7/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3730 - val_loss: 0.3761
Epoch 8/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3596 - val_loss: 0.3707
Epoch 9/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3639 - val_loss: 0.3661
Epoch 10/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3537 - val_loss: 0.3622
Epoch 11/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3498 - val_loss: 0.3587
Epoch 12/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

In [12]:
scaler_y.inverse_transform(y_pred)

array([[200892.1 ],
       [149106.56],
       [197195.33]], dtype=float32)

In [13]:
scaler_y.inverse_transform(y_test[:3])

array([[245800.],
       [137900.],
       [218200.]])

In [14]:
output = keras.layers.Dense(1, name="main_output")(concat)
aux_output = keras.layers.Dense(1, name="aux_output")(hidden2)
model = keras.Model(inputs=[input_A, input_B], outputs=[output,
aux_output])

In [15]:
model.compile(loss=["mse", "mse"], loss_weights=[0.9, 0.1],
optimizer="sgd")

In [16]:
history = model.fit(
[X_train_A, X_train_B], [y_train, y_train], epochs=20,
validation_data=([X_valid_A, X_valid_B], [y_valid, y_valid]))

Epoch 1/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.6024 - loss: 0.5049 - main_output_loss: 0.4941 - val_aux_output_loss: 0.4658 - val_loss: 0.3819 - val_main_output_loss: 0.3724
Epoch 2/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.4542 - loss: 0.3710 - main_output_loss: 0.3618 - val_aux_output_loss: 0.4322 - val_loss: 0.3461 - val_main_output_loss: 0.3364
Epoch 3/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.4177 - loss: 0.3346 - main_output_loss: 0.3253 - val_aux_output_loss: 0.4208 - val_loss: 0.3318 - val_main_output_loss: 0.3218
Epoch 4/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.4164 - loss: 0.3303 - main_output_loss: 0.3208 - val_aux_output_loss: 0.4101 - val_loss: 0.3217 - val_main_output_loss: 0.3118
Epoch 5/20
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.4090 - loss: 0.3201 - main_output_loss: 0.3102 - val_aux_output_loss: 0.4005 - val_loss: 0.3178 - val_main_output_loss: 0.3085


In [17]:
total_loss, main_loss, aux_loss = model.evaluate(
[X_test_A, X_test_B], [y_test, y_test])

160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - aux_output_loss: 0.3620 - loss: 0.3069 - main_output_loss: 0.3008


In [18]:
total_loss, main_loss, aux_loss

(0.3048982620239258, 0.29865604639053345, 0.3612128496170044)

In [19]:
y_pred_main, y_pred_aux = model.predict([X_new_A, X_new_B])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step


In [20]:
scaler_y.inverse_transform(y_pred_main)

array([[221308.94 ],
       [125937.125],
       [184451.39 ]], dtype=float32)

In [21]:
class WideAndDeepModel(keras.Model):
    def __init__(self, units=30, activation="relu", **kwargs):
        super().__init__(**kwargs) # handles standard args (e.g., name)
        self.hidden1 = keras.layers.Dense(units, activation=activation)
        self.hidden2 = keras.layers.Dense(units, activation=activation)
        self.main_output = keras.layers.Dense(1)
        self.aux_output = keras.layers.Dense(1)

    def call(self, inputs):
        input_A, input_B = inputs
        hidden1 = self.hidden1(input_B)
        hidden2 = self.hidden2(hidden1)
        concat = keras.layers.concatenate([input_A, hidden2])
        main_output = self.main_output(concat)
        aux_output = self.aux_output(hidden2)
        return main_output, aux_output

model1 = WideAndDeepModel()

In [22]:
model

<Functional name=functional_2, built=True>

In [23]:
model.save("my_keras_model.keras")

## Saving Model

In [24]:
model = keras.models.load_model("my_keras_model.keras")

In [25]:
checkpoint_cb = keras.callbacks.ModelCheckpoint("my_keras_model.keras")
history = model.fit([X_train_A, X_train_B], [y_train, y_train], epochs=10, callbacks=[checkpoint_cb])

Epoch 1/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - aux_output_loss: 0.3450 - loss: 0.2806 - main_output_loss: 0.2734
Epoch 2/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - aux_output_loss: 0.3488 - loss: 0.2830 - main_output_loss: 0.2757
Epoch 3/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - aux_output_loss: 0.3513 - loss: 0.2868 - main_output_loss: 0.2796
Epoch 4/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - aux_output_loss: 0.3452 - loss: 0.2798 - main_output_loss: 0.2726
Epoch 5/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - aux_output_loss: 0.3384 - loss: 0.2775 - main_output_loss: 0.2707
Epoch 6/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - aux_output_loss: 0.3550 - loss: 0.2853 - main_output_loss: 0.2776
Epoch 7/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - aux_output_loss: 0.3469 - loss: 0.2813 - main_output_loss: 0.2740
Epoch 8/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.3504 - loss: 0.2834 - main_output_loss: 0.2760
Epoch 9/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 

In [29]:
checkpoint_cb = keras.callbacks.ModelCheckpoint("my_keras_model.h5", save_best_only=True)
history = model.fit([X_train_A, X_train_B], [y_train, y_train], epochs=10,
validation_data=([X_valid_A, X_valid_B], [y_valid, y_valid]), callbacks=[checkpoint_cb])
model = keras.models.load_model("my_keras_model.keras")

Epoch 1/10
351/360 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - aux_output_loss: 0.3199 - loss: 0.2648 - main_output_loss: 0.2587

360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.3203 - loss: 0.2651 - main_output_loss: 0.2590 - val_aux_output_loss: 0.3484 - val_loss: 0.2948 - val_main_output_loss: 0.2888
Epoch 2/10
331/360 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - aux_output_loss: 0.3242 - loss: 0.2665 - main_output_loss: 0.2601

360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.3249 - loss: 0.2671 - main_output_loss: 0.2607 - val_aux_output_loss: 0.3446 - val_loss: 0.2837 - val_main_output_loss: 0.2769
Epoch 3/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.3266 - loss: 0.2668 - main_output_loss: 0.2601 - val_aux_output_loss: 0.3447 - val_loss: 0.2846 - val_main_output_loss: 0.2779
Epoch 4/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.3304 - loss: 0.2698 - main_output_loss: 0.2631 - val_aux_output_loss: 0.3447 - val_loss: 0.2852 - val_main_output_loss: 0.2785
Epoch 5/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.3401 - loss: 0.2827 - main_output_loss: 0.2763 - val_aux_output_loss: 0.3447 - val_loss: 0.2862 - val_main_output_loss: 0.2798
Epoch 6/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.3443 - loss: 0.2871 - main_output_loss: 0.2808 - val_aux_output_loss: 0.3447 - val_loss: 0.2915 - val_main_output_loss: 0.2855
Epoch 7/10


360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.3237 - loss: 0.2670 - main_output_loss: 0.2607 - val_aux_output_loss: 0.3422 - val_loss: 0.2827 - val_main_output_loss: 0.2760
Epoch 9/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.3320 - loss: 0.2750 - main_output_loss: 0.2687 - val_aux_output_loss: 0.3427 - val_loss: 0.2840 - val_main_output_loss: 0.2775
Epoch 10/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - aux_output_loss: 0.3228 - loss: 0.2658 - main_output_loss: 0.2594 - val_aux_output_loss: 0.3450 - val_loss: 0.2883 - val_main_output_loss: 0.2820
